In [1]:
# Setup
import math_toolkit
math_toolkit.activate()

# Heat equation Fourier animation

The heat equation describes a function that smooths itself out over time:

$$
\frac{\partial u}{\partial t}=\kappa\frac{\partial^2 u}{\partial x^2}.
$$

On a 1-periodic interval, Fourier modes are the natural coordinates. A cosine or sine mode with frequency $n$ is an eigenfunction of the second derivative, so the heat equation does not mix the modes. It only changes their sizes.

This worksheet solves the periodic heat equation by Fourier coefficients, then animates what happens to two kinds of initial data: a rough Fourier series and a square wave.

## The Fourier solution

Suppose the initial temperature is

$$
u(x,0)=a_0+\sum_{n=1}^{\infty}\left(a_n\cos(2\pi n x)+b_n\sin(2\pi n x)\right).
$$

The heat equation gives each frequency its own decay factor:

$$
u(x,t)=a_0+\sum_{n=1}^{\infty}e^{-4\pi^2\kappa n^2t}\left(a_n\cos(2\pi n x)+b_n\sin(2\pi n x)\right).
$$

The constant term stays fixed. Higher frequencies disappear much faster because of the $n^2$ in the exponent.

## In this notebook the plotted time variable $t$ is an ordinary `math_toolkit`
parameter. Press the play button beside the $t$ slider to run the
Python-owned animation API.

No discrete FFT is used below. We either write the Fourier coefficients down
as a finite symbolic series or compute them from the defining integrals.


## rough_modes = [
    (1, 0.00, 0.75),
    (2, -0.35, 0.00),
    (5, 0.10, -0.14),
    (9, -0.06, 0.07),
    (14, 0.04, 0.05),
    (21, -0.03, 0.04),
    (32, 0.02, -0.03),
]

rough_initial = 0
rough_heat = 0
kappa_rough = Rational(3, 200)
for mode, cos_coefficient, sin_coefficient in rough_modes:
    spatial_mode = (
        cos_coefficient * cos(2 * pi * mode * x)
        + sin_coefficient * sin(2 * pi * mode * x)
    )
    rough_initial += spatial_mode
    rough_heat += exp(-4 * pi**2 * kappa_rough * mode**2 * t) * spatial_mode

fig1 = figure("Rough Fourier data under the heat equation", new=True)
fig1.view.home_range = {"x": (-1/2, 1/2), "y": (-1.4, 1.4)}
fig1.show()

fig1.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 5,
        "step": 0.01,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 1,
    },
})

fig1.plot(
    rough_initial,
    x,
    name="initial data",
    label=r"$u(x,0)$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=1201,
)
fig1.plot(
    rough_heat,
    x,
    name="heat solution",
    label=r"$u(x,t)$",
    style={"color": "#2563eb", "width": 3},
    samples=1201,
)

## :::{admonition} Question
Which features disappear almost immediately? Which features are still visible after the solution has already become much smoother?
:::

## Frequency damping

The animation above is not doing time stepping. It multiplies mode $n$ by

$$
e^{-4\pi^2\kappa n^2t}.
$$

The next plot shows that decay factor for a few different times.

In [3]:
## fig2 = figure("Heat-equation damping by Fourier mode", new=True)
fig2.view.home_range = {"x": (0, 40), "y": (-0.02, 1.05)}
fig2.show()

for time_value, color in [
    (0, "#111827"),
    (Rational(1, 100), "#2563eb"),
    (Rational(1, 25), "#16a34a"),
    (Rational(3, 25), "#dc2626"),
]:
    fig2.list_plot(
        exp(-4 * pi**2 * kappa_rough * n**2 * time_value),
        (n, 0, 41),
        name=f"t={float(time_value):g}",
        label=rf"$t={float(time_value):g}$",
        style={"color": color},
    )

## Square-wave initial data

Now start with the same discontinuous shape from the Fourier inversion notebook: one half of the period is hot and the other half is cold.

The heat equation instantly rounds the jump. Unlike Fourier partial sums, the heat evolution is not trying to preserve a sharp discontinuity with finitely many modes. It is genuinely changing the function.

In [4]:
## Sq = Function("Sq")(x) @EqDef@ (
    If(x > 0).Then(1)
    .If(x < 0).Then(-1)
    .Otherwise(0)
)

FT_a_Sq_0 = Integral(Sq(s), (s, -Rational(1, 2), Rational(1, 2)))
FT_a_Sq = Function("FT_a_Sq", latex=r"a")(n) @EqDef@ (
    2 * Integral(
        Sq(s) * cos(2 * pi * n * s),
        (s, -Rational(1, 2), Rational(1, 2)),
    )
)
FT_b_Sq = Function("FT_b_Sq", latex=r"b")(n) @EqDef@ (
    2 * Integral(
        Sq(s) * sin(2 * pi * n * s),
        (s, -Rational(1, 2), Rational(1, 2)),
    )
)

kappa_square = Rational(3, 500)
H_Sq = Function("H_Sq")(x, t, N) @EqDef@ (
    FT_a_Sq_0
    + Sum(
        exp(-4 * pi**2 * kappa_square * n**2 * t)
        * (
            FT_a_Sq(n) * cos(2 * pi * n * x)
            + FT_b_Sq(n) * sin(2 * pi * n * x)
        ),
        (n, 1, N),
    )
)

r"""
The square-wave heat solution below uses the same integral-defined Fourier
coefficients as the Fourier inversion notebook, but each mode is multiplied
by the heat factor $e^{-4\pi^2 \kappa n^2 t}$.
""" >> md

fig3 = figure("Square wave under the heat equation", new=True)
fig3.view.home_range = {"x": (-1/2, 1/2), "y": (-1.25, 1.25)}
fig3.show()

fig3.parameters({
    t: {
        "value": 0.0,
        "min": 0.0,
        "max": 1,
        "step": 0.01,
        "animated": True,
        "animation_mode": "forward",
        "animation_rate_hz": 20,
        "animation_speed": 0.5,
    },
    N: {"value": 80, "min": 1, "max": 160, "step": 1, "animated": False},
})

fig3.plot(
    Sq(x),
    x,
    name="initial square wave",
    label=r"$u(x,0)$",
    style={"color": "#64748b", "width": 3, "dash": "dot"},
    samples=2001,
)
fig3.plot(
    H_Sq(x, t, N),
    x,
    name="heat solution",
    label=r"$u_N(x,t)$",
    style={"color": "#dc2626", "width": 3},
    samples=2001,
)


The square-wave heat solution below uses the same integral-defined Fourier
coefficients as the Fourier inversion notebook, but each mode is multiplied
by the heat factor $e^{-4\pi^2 \kappa n^2 t}$.


## :::{admonition} Question
Where does the square wave change first? What happens to the average temperature as time passes?
:::

## Now you try

Change `kappa_rough` or `kappa_square` in one of the animation cells. Larger values make heat diffuse faster. Smaller values make the same movie feel closer to slow motion.

Then change the initial data. For example, add another high-frequency sine or cosine mode to `rough_modes`, or adjust the square-wave cutoff `N` and watch what lasts.

In [5]:
## Experiment with your own initial temperature here.
